In [1]:
import os
# Change to MultiBench directory
os.chdir('MultiBench')

In [2]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

h:\Programs\Anaconda\envs\multimodal\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load text modality
with open('data/exported_modalities/mosi_text.pkl', 'rb') as f:
    text_data = pickle.load(f)

print("✓ Text modality loaded")
print(f"  Train: {text_data['train']['data'].shape}")
print(f"  Valid: {text_data['valid']['data'].shape}")
print(f"  Test:  {text_data['test']['data'].shape}")

✓ Text modality loaded
  Train: (1283, 50, 300)
  Valid: (214, 50, 300)
  Test:  (686, 50, 300)


In [4]:
# Convert 7-class labels to 5-class labels
# Original: -3, -2, -1, 0, +1, +2, +3
# TFN 5-class mapping: [-3,-2] → 0, [-1] → 1, [0] → 2, [+1] → 3, [+2,+3] → 4

def convert_to_5class(labels):
    """
    Convert continuous sentiment scores to 5 classes:
    Class 0: Highly Negative (sentiment ≤ -1.5)
    Class 1: Negative (-1.5 < sentiment ≤ -0.5)
    Class 2: Neutral (-0.5 < sentiment ≤ 0.5)
    Class 3: Positive (0.5 < sentiment ≤ 1.5)
    Class 4: Highly Positive (sentiment > 1.5)
    """
    labels = labels.flatten()
    converted = np.zeros_like(labels, dtype=np.int64)
    
    converted[labels <= -1.5] = 0  # Highly Negative
    converted[(labels > -1.5) & (labels <= -0.5)] = 1  # Negative
    converted[(labels > -0.5) & (labels <= 0.5)] = 2  # Neutral
    converted[(labels > 0.5) & (labels <= 1.5)] = 3  # Positive
    converted[labels > 1.5] = 4  # Highly Positive
    
    return converted

# Convert labels
train_labels_5class = convert_to_5class(text_data['train']['labels'])
valid_labels_5class = convert_to_5class(text_data['valid']['labels'])
test_labels_5class = convert_to_5class(text_data['test']['labels'])

# Display class distribution
class_names = ['Highly Negative', 'Negative', 'Neutral', 'Positive', 'Highly Positive']
print("\n5-Class Label Distribution:")
print("=" * 80)

for split_name, labels in [('TRAIN', train_labels_5class), 
                           ('VALID', valid_labels_5class), 
                           ('TEST', test_labels_5class)]:
    print(f"\n{split_name}:")
    for class_idx, class_name in enumerate(class_names):
        count = int(np.sum(labels == class_idx))
        percentage = (count / len(labels)) * 100
        print(f"  Class {class_idx} ({class_name:<18}): {count:>4} samples ({percentage:>5.1f}%)")

print("\n✓ Labels converted to 5 classes")


5-Class Label Distribution:

TRAIN:
  Class 0 (Highly Negative   ):  223 samples ( 17.4%)
  Class 1 (Negative          ):  241 samples ( 18.8%)
  Class 2 (Neutral           ):  233 samples ( 18.2%)
  Class 3 (Positive          ):  229 samples ( 17.8%)
  Class 4 (Highly Positive   ):  357 samples ( 27.8%)

VALID:
  Class 0 (Highly Negative   ):   33 samples ( 15.4%)
  Class 1 (Negative          ):   26 samples ( 12.1%)
  Class 2 (Neutral           ):   48 samples ( 22.4%)
  Class 3 (Positive          ):   39 samples ( 18.2%)
  Class 4 (Highly Positive   ):   68 samples ( 31.8%)

TEST:
  Class 0 (Highly Negative   ):  202 samples ( 29.4%)
  Class 1 (Negative          ):  148 samples ( 21.6%)
  Class 2 (Neutral           ):  103 samples ( 15.0%)
  Class 3 (Positive          ):  114 samples ( 16.6%)
  Class 4 (Highly Positive   ):  119 samples ( 17.3%)

✓ Labels converted to 5 classes


In [5]:
# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Create PyTorch datasets
train_dataset = TensorDataset(
    torch.FloatTensor(text_data['train']['data']),
    torch.LongTensor(train_labels_5class)
)
valid_dataset = TensorDataset(
    torch.FloatTensor(text_data['valid']['data']),
    torch.LongTensor(valid_labels_5class)
)
test_dataset = TensorDataset(
    torch.FloatTensor(text_data['test']['data']),
    torch.LongTensor(test_labels_5class)
)

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("✓ DataLoaders created")
print(f"  Batch size: {batch_size}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Valid batches: {len(valid_loader)}")
print(f"  Test batches: {len(test_loader)}")

✓ DataLoaders created
  Batch size: 64
  Train batches: 20
  Valid batches: 4
  Test batches: 11


In [6]:
# Text Unimodal Model for 5-class Classification (TFN Architecture)
class TextUnimodal5Class(nn.Module):
    def __init__(self, input_dim=300, lstm_hidden=128, embed_dim=128, num_classes=5, dropout=0.2):
        super(TextUnimodal5Class, self).__init__()
        
        # 1. LSTM for temporal modeling
        self.lstm = nn.LSTM(input_dim, lstm_hidden, batch_first=True, bidirectional=False)
        self.lstm_ln = nn.LayerNorm(lstm_hidden)
        
        # 2. Language Embedding Subnetwork
        self.embed_net = nn.Sequential(
            nn.Linear(lstm_hidden, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # 3. Sentiment Inference Subnetwork - 5-class classification
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, num_classes)  # 5 classes output
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight_ih' in name:
                        nn.init.xavier_uniform_(param.data)
                    elif 'weight_hh' in name:
                        nn.init.orthogonal_(param.data)
                    elif 'bias' in name:
                        nn.init.zeros_(param.data)
    
    def forward(self, x):
        # LSTM encoding
        lstm_out, (h_n, c_n) = self.lstm(x)
        last_hidden = self.lstm_ln(h_n[-1])
        
        # Language embedding
        z_l = self.embed_net(last_hidden)
        
        # 5-class classification (logits)
        output = self.classifier(z_l)
        
        return output

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TextUnimodal5Class(input_dim=300, lstm_hidden=128, embed_dim=128, num_classes=5, dropout=0.2)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, 
                                                  patience=3, verbose=True, min_lr=1e-6)

print("✓ Model initialized for 5-class classification")
print(f"  Device: {device}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Loss: CrossEntropyLoss")
print(f"  Optimizer: AdamW with ReduceLROnPlateau")

✓ Model initialized for 5-class classification
  Device: cuda
  Parameters: 279,429
  Loss: CrossEntropyLoss
  Optimizer: AdamW with ReduceLROnPlateau


In [7]:
# Model Summary using torchinfo (better for LSTM models)
try:
    from torchinfo import summary
    
    seq_len = text_data['train']['data'].shape[1]  
    input_dim = text_data['train']['data'].shape[2]  
    batch_size_summary = 2  # for summary display
    
    print("=" * 70)
    print("MODEL SUMMARY")
    print("=" * 70)
    
    summary(model, 
            input_size=(batch_size_summary, seq_len, input_dim),
            device=str(device),
            col_names=["input_size", "output_size", "num_params", "trainable"],
            row_settings=["var_names"],
            verbose=1)
    
except ImportError:
    print("⚠ torchinfo not installed. Installing...")
    print("Run: pip install torchinfo")
    print("\nAlternatively, here's a manual parameter count:")
    print("=" * 70)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\nTotal Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Non-trainable Parameters: {total_params - trainable_params:,}")
    
    print("\nLayer-wise Parameter Count:")
    print("-" * 70)
    for name, module in model.named_children():
        params = sum(p.numel() for p in module.parameters())
        print(f"{name:<20} {params:>15,} parameters")
    print("=" * 70)

MODEL SUMMARY
Layer (type (var_name))                  Input Shape               Output Shape              Param #                   Trainable
TextUnimodal5Class (TextUnimodal5Class)  [2, 50, 300]              [2, 5]                    --                        True
├─LSTM (lstm)                            [2, 50, 300]              [2, 50, 128]              220,160                   True
├─LayerNorm (lstm_ln)                    [2, 128]                  [2, 128]                  256                       True
├─Sequential (embed_net)                 [2, 128]                  [2, 128]                  --                        True
│    └─Linear (0)                        [2, 128]                  [2, 128]                  16,512                    True
│    └─BatchNorm1d (1)                   [2, 128]                  [2, 128]                  256                       True
│    └─ReLU (2)                          [2, 128]                  [2, 128]                  --                  

In [7]:
# Training and evaluation functions
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch_data, batch_labels in loader:
        batch_data = batch_data.to(device)
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_data)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch_data, batch_labels in loader:
            batch_data = batch_data.to(device)
            batch_labels = batch_labels.to(device)
            
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy, all_preds, all_labels

print("✓ Training functions defined")

✓ Training functions defined


In [8]:
# Training loop
num_epochs = 100
patience = 15
best_val_acc = 0
patience_counter = 0

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'lr': []
}

print("=" * 70)
print("TRAINING TEXT UNIMODAL MODEL (5-CLASS CLASSIFICATION)")
print("=" * 70)
print(f"Epochs: {num_epochs} | Batch Size: {batch_size} | Initial LR: 1e-3")
print(f"Target: Paper Accuracy = 38.5%")
print("=" * 70)

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc, _, _ = evaluate(model, valid_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} ({train_acc*100:.1f}%) | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} ({val_acc*100:.1f}%) | "
              f"LR: {current_lr:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc
        }, 'best_text_5class.pt')
        patience_counter = 0
        print(f"  ✓ New best model saved! (Val Acc: {val_acc:.4f} = {val_acc*100:.1f}%)")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break
    
    if current_lr < 1e-6:
        print(f"\nStopping: Learning rate too small ({current_lr:.2e})")
        break

print("=" * 70)
print(f"✓ Training completed!")
print(f"  Best Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.1f}%)")
print(f"  Total Epochs: {epoch + 1}")
print("=" * 70)

TRAINING TEXT UNIMODAL MODEL (5-CLASS CLASSIFICATION)
Epochs: 100 | Batch Size: 64 | Initial LR: 1e-3
Target: Paper Accuracy = 38.5%
Epoch [  1/100] | Train Loss: 1.7416 Acc: 0.2633 (26.3%) | Val Loss: 1.5840 Acc: 0.3037 (30.4%) | LR: 0.001000
  ✓ New best model saved! (Val Acc: 0.3037 = 30.4%)
Epoch [  1/100] | Train Loss: 1.7416 Acc: 0.2633 (26.3%) | Val Loss: 1.5840 Acc: 0.3037 (30.4%) | LR: 0.001000
  ✓ New best model saved! (Val Acc: 0.3037 = 30.4%)
  ✓ New best model saved! (Val Acc: 0.3178 = 31.8%)
  ✓ New best model saved! (Val Acc: 0.3178 = 31.8%)
Epoch [  5/100] | Train Loss: 1.3168 Acc: 0.4484 (44.8%) | Val Loss: 1.4780 Acc: 0.3692 (36.9%) | LR: 0.001000
  ✓ New best model saved! (Val Acc: 0.3692 = 36.9%)
Epoch [  5/100] | Train Loss: 1.3168 Acc: 0.4484 (44.8%) | Val Loss: 1.4780 Acc: 0.3692 (36.9%) | LR: 0.001000
  ✓ New best model saved! (Val Acc: 0.3692 = 36.9%)
Epoch 00009: reducing learning rate of group 0 to 5.0000e-04.
Epoch 00009: reducing learning rate of group 0 to

In [9]:
# Load best model and evaluate on test set
checkpoint = torch.load('best_text_5class.pt')
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

# TFN Paper benchmark for 5-class
paper_acc = 0.385  # 38.5%

print("\n" + "=" * 70)
print("TEST SET RESULTS (Text Unimodal - 5-Class Classification)")
print("=" * 70)
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("=" * 70)

print("\nComparison with TFN Paper (Zadeh et al., 2017):")
print("-" * 70)
print(f"{'Metric':<20} {'Our Model':<20} {'Paper':<20} {'Difference':<15}")
print("-" * 70)
print(f"{'Accuracy':<20} {test_acc:.4f} ({test_acc*100:.1f}%){'':<6} {paper_acc:.4f} ({paper_acc*100:.1f}%){'':<6} {(test_acc-paper_acc)*100:+.1f}%")
print("-" * 70)

if abs(test_acc - paper_acc) < 0.03:
    print("✓ Results are within expected range of paper benchmark!")
elif test_acc > paper_acc:
    print("✓ Results exceed paper benchmark!")
else:
    print("⚠ Results differ from paper. Consider adjusting hyperparameters.")

print("\n✓ 5-Class evaluation completed!")


TEST SET RESULTS (Text Unimodal - 5-Class Classification)
Test Accuracy: 0.3397 (33.97%)

Comparison with TFN Paper (Zadeh et al., 2017):
----------------------------------------------------------------------
Metric               Our Model            Paper                Difference     
----------------------------------------------------------------------
Accuracy             0.3397 (34.0%)       0.3850 (38.5%)       -4.5%
----------------------------------------------------------------------
⚠ Results differ from paper. Consider adjusting hyperparameters.

✓ 5-Class evaluation completed!
